In [1]:
# import libraries
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
# File paths

DATA_DIR = Path("C:/Users/htunlong/OneDrive - UGent/Research/Postdoc/Fundings/VMM/Data/Data_Module_5")

MPS_ANCHOR_FILE = DATA_DIR / "Module 5/M5_step01_MPS_spatial_anchor.xlsx"
ICM_LOCATIONS_FILE = DATA_DIR / "Module 2/ICM_Meetlocaties_herzien_VMM2 (1).xlsx"
for path in [MPS_ANCHOR_FILE, ICM_LOCATIONS_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

In [3]:
# Read Step 1 output and Module 2 location file

gpkg_candidates = pd.read_excel(
    MPS_ANCHOR_FILE,
    sheet_name="MPS_Spatial_Anchor",
)

icm_raw = pd.read_excel(
    ICM_LOCATIONS_FILE,
    sheet_name="Blad1",
)

coverage = pd.read_excel(
    MPS_ANCHOR_FILE,
    sheet_name="MPS_Coverage_Check",
)
    
MPS_LOCATION_LOOKUP = {
    "MP01": {
        "location_name": "Goudbergstraat",
        "watercourse": "Dommel",
    },
    "MP02": {
        "location_name": "Hoksentstraat",
        "watercourse": "Dommel",
    },
    "MP03": {
        "location_name": "Watermolen van Molhem",
        "watercourse": "Dommel",
    },
    "MP04": {
        "location_name": "Warmbeek downstream of canal",
        "watercourse": "Warmbeek",
    },
    "MP06": {
        "location_name": "Warmbeek upstream of Prinsenloop",
        "watercourse": "Warmbeek",
    },
    "MP07": {
        "location_name": "Eindergatloop",
        "watercourse": "Eindergatloop",
    },
}
    

print("MPS anchor shape:", gpkg_candidates.shape)
print("ICM raw location shape:", icm_raw.shape)
print("MPS anchor shape:", coverage.shape)

print("\nMPS stations:")
print(gpkg_candidates["mps_station_id"].tolist())

print("\nICM columns:")
print(icm_raw.columns.tolist())

MPS anchor shape: (11, 12)
ICM raw location shape: (18, 10)
MPS anchor shape: (6, 9)

MPS stations:
['MP01', 'MP01', 'MP02', 'MP02', 'MP03', 'MP03', 'MP04', 'MP04', 'MP06', 'MP06', 'MP07']

ICM columns:
['X', 'Y', 'id', 'NaamLoc', 'icmLink', 'ICMsuffix', 'upstrORdownstr', 'ICM_x', 'ICM_y', 'waterloop']


In [4]:
icm_raw.head()

,X,Y,id,NaamLoc,icmLink,ICMsuffix,upstrORdownstr,ICM_x,ICM_y,waterloop
0,225253.683765,221603.575669,1,I_D1,DOMN011,1,afw,225292,221559,Dommel
1,227336.704709,220693.996706,2,I_W1,PRI031,1,opw,227287,220608,Warmbeek
2,227868.546409,221585.329947,3,I_W2,WAR006c,1,afw,227840,221704,Warmbeek
3,223380.743415,214362.488609,5,I_D2,EIN017,1,afw,223408,214357,Dommel
4,223439.358568,213501.766797,6,I_D3,HOLk006d,1,opw,223443,213500,Dommel


In [5]:
# ICM link column includes information of the location.
# upstrORdownstr column includes information of the up/downstream 

In [6]:
# Clean data
def clean_text(value):
    """
    Convert a value to a clean string.
    Missing values become an empty string.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


In [7]:
# classify ICM link column 


def classify_link_prefix(icm_link):
    """
    Classify an ICM link based on its prefix.

    This is a practical first classification.
    It helps us select candidate hydraulic locations for each MPS station.
    """
    link = clean_text(icm_link).upper()

    if link.startswith("DOM"):
        return "Dommel main channel"
    if link.startswith("WAR"):
        return "Warmbeek main channel"
    if link.startswith("EIN"):
        return "Eindergatloop tributary"
    if link.startswith("PRI"):
        return "Prinsenloop tributary"
    if link.startswith("HOL"):
        return "Holvenloop tributary"
    if link.startswith("PEE"):
        return "Peerdeloop tributary"
    if link.startswith("BOL"):
        return "Bollisenbeek tributary"

    return "Unknown"



In [8]:
# classify up/downstream column

def side_label(value):
    """
    Translate the Dutch upstream/downstream code.

    opw = opwaarts = upstream
    afw = afwaarts = downstream
    """
    value = clean_text(value).lower()

    if value == "opw":
        return "upstream"
    if value == "afw":
        return "downstream"

    return "unknown"

In [9]:
# make the summary table

def prepare_icm_locations_table(icm_df):
    """
    Prepare a clean Module 2 hydraulic-location catalogue.
    """
    required_columns = [
        "NaamLoc",
        "icmLink",
        "ICMsuffix",
        "upstrORdownstr",
        "ICM_x",
        "ICM_y",
        "waterloop",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in icm_df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing expected columns: {missing_columns}")

    result = icm_df.copy()

    result["icm_link"] = result["icmLink"].map(clean_text)
    result["icm_series_column"] = (
        result["icm_link"] + "." + result["ICMsuffix"].astype(str)
    )

    result["hydraulic_location_id"] = result["NaamLoc"].map(clean_text)
    result["upstream_downstream_code"] = result["upstrORdownstr"].map(clean_text)
    result["upstream_downstream_label"] = result["upstrORdownstr"].map(side_label)
    result["watercourse_from_file"] = result["waterloop"].map(clean_text)
    result["link_group"] = result["icm_link"].map(classify_link_prefix)

    keep_columns = [
        "hydraulic_location_id",
        "icm_series_column",
        "icm_link",
        "ICMsuffix",
        "upstream_downstream_code",
        "upstream_downstream_label",
        "ICM_x",
        "ICM_y",
        "watercourse_from_file",
        "link_group",
    ]

    result = result[keep_columns]

    result = result.sort_values(
        [
            "watercourse_from_file",
            "link_group",
            "hydraulic_location_id",
            "icm_series_column",
        ]
    ).reset_index(drop=True)

    return result


In [10]:
icm_locations = prepare_icm_locations_table(icm_raw)

print("Prepared ICM locations:", icm_locations.shape)
icm_locations

Prepared ICM locations: (18, 10)


,hydraulic_location_id,icm_series_column,icm_link,ICMsuffix,upstream_downstream_code,upstream_downstream_label,ICM_x,ICM_y,watercourse_from_file,link_group
0,I_D6,BOLk058d.1,BOLk058d,1,opw,upstream,223772,205512,Dommel,Bollisenbeek tributary
1,D8,DOM461!.1,DOM461!,1,opw,upstream,223973,205673,Dommel,Dommel main channel
2,D9,DOM299.1,DOM299,1,opw,upstream,224751,210268,Dommel,Dommel main channel
3,I_D1,DOMN011.1,DOMN011,1,afw,downstream,225292,221559,Dommel,Dommel main channel
4,I_D4,DOM192A.1,DOM192A,1,opw,upstream,224429,213475,Dommel,Dommel main channel
5,I_D7,DOM531!.1,DOM531!,1,afw,downstream,223882,205448,Dommel,Dommel main channel
6,L11_22,DOM154Ac1.1,DOM154Ac1,1,afw,downstream,223547,213792,Dommel,Dommel main channel
7,L11_23,DOM559!.1,DOM559!,1,opw,upstream,224565,203240,Dommel,Dommel main channel
8,I_D2,EIN017.1,EIN017,1,afw,downstream,223408,214357,Dommel,Eindergatloop tributary
9,I_D3,HOLk006d.1,HOLk006d,1,opw,upstream,223443,213500,Dommel,Holvenloop tributary


In [11]:
# MAPPING with MPS station using the channel information
# These rules are deliberately conservative.
# They do not pick one final hydraulic node.
# They only say which Module 2 hydraulic groups are plausible candidates.

MPS_TO_HYDRAULIC_GROUPS = {
    "MP01": ["Dommel main channel"],
    "MP02": ["Dommel main channel"],
    "MP03": ["Dommel main channel"],
    "MP04": ["Warmbeek main channel"],

    # MP06 is Warmbeek upstream of Prinsenloop.
    # For now we keep both Warmbeek and Prinsenloop candidates.
    "MP06": ["Warmbeek main channel", "Prinsenloop tributary"],

    # MP07 is explicitly Eindergatloop.
    "MP07": ["Eindergatloop tributary"],
}


In [12]:
# Build the mapping table
def build_updated_anchor_table(coverage_table, location_lookup, coordinate_candidates):
    """
    Build one anchor row per MPS station.

    Coordinates are filled only if there is exactly one coordinate candidate.
    """
    records = []

    for _, row in coverage_table.iterrows():
        station_id = row["mps_station_id"]
        lookup = location_lookup[station_id]

        candidates = coordinate_candidates[
            coordinate_candidates["mps_station_id"] == station_id
        ]

        if len(candidates) == 1:
            chosen = candidates.iloc[0]

            coordinate_status = "filled_from_unique_watercourse_match"
            gpkg_station_name = chosen["gpkg_station_name"]
            gpkg_station_no = chosen["gpkg_station_no"]
            longitude = chosen["longitude"]
            latitude = chosen["latitude"]
            x_lambert72 = chosen["x_lambert72"]
            y_lambert72 = chosen["y_lambert72"]

        elif len(candidates) > 1:
            coordinate_status = "ambiguous_candidates_manual_confirmation_needed"
            gpkg_station_name = pd.NA
            gpkg_station_no = pd.NA
            longitude = pd.NA
            latitude = pd.NA
            x_lambert72 = pd.NA
            y_lambert72 = pd.NA

        else:
            coordinate_status = "no_candidate_found"
            gpkg_station_name = pd.NA
            gpkg_station_no = pd.NA
            longitude = pd.NA
            latitude = pd.NA
            x_lambert72 = pd.NA
            y_lambert72 = pd.NA

        records.append({
            "mps_station_id": station_id,
            "location_name": lookup["location_name"],
            "watercourse": lookup["watercourse"],
            "gpkg_station_name": gpkg_station_name,
            "gpkg_station_no": gpkg_station_no,
            "longitude": longitude,
            "latitude": latitude,
            "x_lambert72": x_lambert72,
            "y_lambert72": y_lambert72,
            "mapped_river_segment_id": pd.NA,
            "mapped_hydraulic_node": pd.NA,
            "mapped_pegase_segment": pd.NA,
            "n_raw_channels": row["n_raw_channels"],
            "n_rows_with_any_value": row["n_rows_with_any_value"],
            "first_observation": row["first_observation"],
            "last_observation": row["last_observation"],
            "coverage_pct_within_station_period": row["coverage_pct_within_station_period"],
            "gaps_gt_10min30s_count": row["gaps_gt_10min30s_count"],
            "max_gap_hours": row["max_gap_hours"],
            "coordinate_candidate_count": len(candidates),
            "coordinate_status": coordinate_status,
        })

    return pd.DataFrame(records)


updated_anchor = build_updated_anchor_table(
    coverage,
    MPS_LOCATION_LOOKUP,
    gpkg_candidates,
)

updated_anchor[
    [
        "mps_station_id",
        "location_name",
        "watercourse",
        "gpkg_station_name",
        "longitude",
        "latitude",
        "coordinate_candidate_count",
        "coordinate_status",
    ]
]

,mps_station_id,location_name,watercourse,gpkg_station_name,longitude,latitude,coordinate_candidate_count,coordinate_status
0,MP01,Goudbergstraat,Dommel,<NA>,<NA>,<NA>,2,ambiguous_candidates_manual_confirmation_needed
1,MP02,Hoksentstraat,Dommel,<NA>,<NA>,<NA>,2,ambiguous_candidates_manual_confirmation_needed
2,MP03,Watermolen van Molhem,Dommel,<NA>,<NA>,<NA>,2,ambiguous_candidates_manual_confirmation_needed
3,MP04,Warmbeek downstream of canal,Warmbeek,<NA>,<NA>,<NA>,2,ambiguous_candidates_manual_confirmation_needed
4,MP06,Warmbeek upstream of Prinsenloop,Warmbeek,<NA>,<NA>,<NA>,2,ambiguous_candidates_manual_confirmation_needed
5,MP07,Eindergatloop,Eindergatloop,Pelt/Rallylaan/Eindergatloop,5.409227,51.235524,1,filled_from_unique_watercourse_match


In [13]:
# Update module 2 using coordinates

In [14]:
def clean_text(value):
    """
    Convert a value to clean text.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def classify_link_prefix(icm_link):
    """
    Classify ICM links into watercourse groups.
    """
    link = clean_text(icm_link).upper()

    if link.startswith("DOM"):
        return "Dommel main channel"
    if link.startswith("WAR"):
        return "Warmbeek main channel"
    if link.startswith("EIN"):
        return "Eindergatloop tributary"
    if link.startswith("PRI"):
        return "Prinsenloop tributary"
    if link.startswith("HOL"):
        return "Holvenloop tributary"
    if link.startswith("PEE"):
        return "Peerdeloop tributary"
    if link.startswith("BOL"):
        return "Bollisenbeek tributary"

    return "Unknown"


def side_label(value):
    """
    Translate Dutch upstream/downstream labels.
    """
    value = clean_text(value).lower()

    if value == "opw":
        return "upstream"
    if value == "afw":
        return "downstream"

    return "unknown"


def prepare_icm_locations_table(icm_df):
    """
    Prepare clean ICM hydraulic locations.
    """
    result = icm_df.copy()

    result["icm_link"] = result["icmLink"].map(clean_text)
    result["icm_series_column"] = result["icm_link"] + "." + result["ICMsuffix"].astype(str)
    result["hydraulic_location_id"] = result["NaamLoc"].map(clean_text)
    result["upstream_downstream_label"] = result["upstrORdownstr"].map(side_label)
    result["watercourse_from_file"] = result["waterloop"].map(clean_text)
    result["link_group"] = result["icm_link"].map(classify_link_prefix)

    return result[
        [
            "hydraulic_location_id",
            "icm_series_column",
            "icm_link",
            "upstream_downstream_label",
            "ICM_x",
            "ICM_y",
            "watercourse_from_file",
            "link_group",
        ]
    ].copy()


MPS_TO_HYDRAULIC_GROUPS = {
    "MP01": ["Dommel main channel"],
    "MP02": ["Dommel main channel"],
    "MP03": ["Dommel main channel"],
    "MP04": ["Warmbeek main channel"],
    "MP06": ["Warmbeek main channel", "Prinsenloop tributary"],
    "MP07": ["Eindergatloop tributary"],
}


def build_coordinate_based_hydraulic_candidates(gpkg_candidates, icm_locations, mapping_rules):
    """
    For each possible MPS coordinate, rank possible ICM locations by distance.

    Distances are in metres because both datasets are converted to EPSG:31370.
    """
    records = []

    for _, mps_candidate in gpkg_candidates.iterrows():
        station_id = mps_candidate["mps_station_id"]
        allowed_groups = mapping_rules.get(station_id, [])

        icm_subset = icm_locations[
            icm_locations["link_group"].isin(allowed_groups)
        ].copy()

        for _, icm_row in icm_subset.iterrows():
            dx = float(mps_candidate["x_lambert72"]) - float(icm_row["ICM_x"])
            dy = float(mps_candidate["y_lambert72"]) - float(icm_row["ICM_y"])

            distance_m = (dx**2 + dy**2) ** 0.5

            records.append({
                "mps_station_id": station_id,
                "mps_location_name": mps_candidate["mps_location_name"],
                "mps_watercourse": mps_candidate["mps_watercourse"],
                "gpkg_station_name": mps_candidate["gpkg_station_name"],
                "gpkg_station_no": mps_candidate["gpkg_station_no"],
                "coordinate_match_status": mps_candidate["coordinate_match_status"],
                "hydraulic_location_id": icm_row["hydraulic_location_id"],
                "icm_series_column": icm_row["icm_series_column"],
                "icm_link": icm_row["icm_link"],
                "link_group": icm_row["link_group"],
                "distance_m": round(distance_m, 1),
            })

    candidates = pd.DataFrame(records)

    candidates = candidates.sort_values(
        ["mps_station_id", "gpkg_station_name", "distance_m"]
    ).reset_index(drop=True)

    candidates["distance_rank_for_coordinate_candidate"] = (
        candidates
        .groupby(["mps_station_id", "gpkg_station_no"])
        .cumcount() + 1
    )

    return candidates


icm_raw = pd.read_excel(ICM_LOCATIONS_FILE, sheet_name="Blad1")
icm_locations = prepare_icm_locations_table(icm_raw)

distance_candidates = build_coordinate_based_hydraulic_candidates(
    gpkg_candidates,
    icm_locations,
    MPS_TO_HYDRAULIC_GROUPS,
)

nearest_by_coordinate = distance_candidates[
    distance_candidates["distance_rank_for_coordinate_candidate"] == 1
].copy()

nearest_by_coordinate[
    [
        "mps_station_id",
        "mps_location_name",
        "gpkg_station_name",
        "coordinate_match_status",
        "icm_series_column",
        "link_group",
        "distance_m",
    ]
]

,mps_station_id,mps_location_name,gpkg_station_name,coordinate_match_status,icm_series_column,link_group,distance_m
0,MP01,Goudbergstraat,Peer/Dijkerstraat/Dommel,ambiguous_watercourse_match,DOM559!.1,Dommel main channel,78.4
7,MP01,Goudbergstraat,Pelt/GroteHeide/Dommel,ambiguous_watercourse_match,DOMN011.1,Dommel main channel,3723.1
14,MP02,Hoksentstraat,Peer/Dijkerstraat/Dommel,ambiguous_watercourse_match,DOM559!.1,Dommel main channel,78.4
21,MP02,Hoksentstraat,Pelt/GroteHeide/Dommel,ambiguous_watercourse_match,DOMN011.1,Dommel main channel,3723.1
28,MP03,Watermolen van Molhem,Peer/Dijkerstraat/Dommel,ambiguous_watercourse_match,DOM559!.1,Dommel main channel,78.4
35,MP03,Watermolen van Molhem,Pelt/GroteHeide/Dommel,ambiguous_watercourse_match,DOMN011.1,Dommel main channel,3723.1
42,MP04,Warmbeek downstream of canal,Achel/Beverbekerdijk/Warmbeek,ambiguous_watercourse_match,WAR006c.1,Warmbeek main channel,2097.0
47,MP04,Warmbeek downstream of canal,Pelt/KorteDijk/Warmbeek,ambiguous_watercourse_match,WAR293-WAR292.1,Warmbeek main channel,1214.8
52,MP06,Warmbeek upstream of Prinsenloop,Achel/Beverbekerdijk/Warmbeek,ambiguous_watercourse_match,PRI031.1,Prinsenloop tributary,1470.6
58,MP06,Warmbeek upstream of Prinsenloop,Pelt/KorteDijk/Warmbeek,ambiguous_watercourse_match,WAR293-WAR292.1,Warmbeek main channel,1214.8


In [22]:
OUTPUT_FILE = DATA_DIR / "Module 5/M5_step01_02_updated_with_MPS_coordinates.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    updated_anchor.to_excel(
        writer,
        sheet_name="MPS_Spatial_Anchor_Updated",
        index=False,
    )


    gpkg_candidates.to_excel(
        writer,
        sheet_name="MPS_GPKG_Candidates",
        index=False,
    )

    icm_locations.to_excel(
        writer,
        sheet_name="ICM_Hydraulic_Locations",
        index=False,
    )

    distance_candidates.to_excel(
        writer,
        sheet_name="MPS_ICM_Distance_Candidates",
        index=False,
    )

    nearest_by_coordinate.to_excel(
        writer,
        sheet_name="Nearest_By_Coordinate",
        index=False,
    )

print(f"Saved: {OUTPUT_FILE}")

Saved: C:\Users\htunlong\OneDrive - UGent\Research\Postdoc\Fundings\VMM\Data\Data_Module_5\Module 5\M5_step01_02_updated_with_MPS_coordinates.xlsx
